# Notebook 07 — All in One Runner

Notebook này chạy lần lượt toàn bộ pipeline của dự án: tạo dữ liệu, baseline, chấm mô phỏng, so sánh, và vẽ biểu đồ.

**Lưu ý về nguồn dữ liệu**:
- `src/regenerate_all.py` sẽ tự động gọi `src/harden_hidden_tests.py` ở cuối để làm khó các hidden test cases.
- Để tránh ghi đè các hidden test case đang ổn định hiện tại (ví dụ: bài `task_id 100` đang chạy tốt và không muốn bị lặp lại các hidden test), bước `regenerate_all.py` sẽ tự động được **bỏ qua** nếu hai tệp dữ liệu đã tồn tại trên đĩa.

Mục tiêu là gom các lệnh rải trong nhiều notebook thành một notebook duy nhất để chạy end-to-end.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

cwd = Path.cwd()
if cwd.name in {'notebookes', 'notebooks', 'src', 'data', 'results'}:
    BASE = cwd.parent
else:
    BASE = cwd

sys.path.insert(0, str(BASE / 'src'))
print(f"✓ BASE = {BASE}")
print(f"✓ app.py exists: {(BASE / 'app.py').exists()}")
print(f"✓ src/regenerate_all.py exists: {(BASE / 'src' / 'regenerate_all.py').exists()}")
print(f"✓ src/run_baseline.py exists: {(BASE / 'src' / 'run_baseline.py').exists()}")

In [ ]:
def run_script(script_path: Path) -> None:
    print(f"
=== Running {script_path.name} ===")
    completed = subprocess.run([sys.executable, str(script_path)], cwd=str(BASE), check=True)
    print(f"=== Finished {script_path.name} (return code {completed.returncode}) ===")

# ── Bước 0: Tạo/đồng bộ dữ liệu (bỏ qua nếu đã có file để không ghi đè) ──
hidden_file = BASE / "data" / "processed" / "hidden_v2.json"
subs_file   = BASE / "data" / "processed" / "submissions_50.json"
if hidden_file.exists() and subs_file.exists():
    print("[SKIP] Dữ liệu đã tồn tại — bỏ qua regenerate_all.py")
else:
    run_script(BASE / "src" / "regenerate_all.py")

# ── Pipeline chính (thứ tự đúng) ──
pipeline = [
    BASE / "src" / "run_baseline.py",      # Kiểm chứng solution chuẩn
    BASE / "src" / "run_grading_v2.py",    # Chấm 100 bài nộp (dùng runner_v3)
    BASE / "src" / "comparison_3sets.py",  # So sánh 4 cấu hình → comparison_week4.csv
    BASE / "src" / "generate_plots.py",    # Vẽ biểu đồ từ comparison_week4.csv
]

for script in pipeline:
    if not script.exists():
        raise FileNotFoundError(f"Khong tim thay script: {script}")
    run_script(script)

print("
[DONE] Toàn bộ pipeline hoàn tất!")


In [ ]:
import pandas as pd

# File outputs tuần 4
files_to_check = {
    "Baseline"       : BASE / "results" / "baseline_summary.csv",
    "Grading v2"     : BASE / "results" / "error_analysis_v2.csv",
    "Comparison T4"  : BASE / "results" / "comparison_week4.csv",
}

for name, path in files_to_check.items():
    status = "✓" if path.exists() else "✗ THIẾU"
    print(f"  [{status}] {name}: {path.name}")

# Hiển thị mẫu kết quả comparison tuần 4
csv_path = BASE / "results" / "comparison_week4.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"
Comparison T4: {len(df)} rows")
    cols = [c for c in ["sv_id","task_id","func","topic","actual_error_type",
                         "set1_tpr","set2_tpr","set3_tpr","set4_tpr",
                         "is_fp_set1","is_fp_set4"] if c in df.columns]
    display(df[cols].head(10))


## Kết quả

Sau khi chạy xong notebook này, toàn bộ file kết quả trong `results/` sẽ được cập nhật tự động.